# 04 — Reorder Prediction Modeling

This notebook trains and evaluates models for next-basket reorder prediction. The task is to rank each user's historically purchased products by their likelihood of appearing in the user's next basket.

Two models are compared:

1. Logistic Regression as an interpretable baseline
2. Gradient-Boosted Trees as a stronger nonlinear Spark ML model

Evaluation focuses on both classification quality and recommendation quality using ROC-AUC, PR-AUC, Precision@K, and Recall@K.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [0]:
base_path = "/Volumes/workspace/default/instacart"  # change if your volume path is different

silver_path = f"{base_path}/silver"
gold_path = f"{base_path}/gold"
model_path = f"{base_path}/models"


dbutils.fs.mkdirs(model_path)

In [0]:
df = spark.read.parquet(f"{gold_path}/reorder_training_set")
display(df.limit(10))

1. Check label balance

In [0]:
display(
    df.groupBy("label")
      .agg(F.count("*").alias("rows"))
      .withColumn("pct", F.col("rows") / F.sum("rows").over(Window.partitionBy()))
)

In [0]:
positive_rate = df.agg(F.avg("label")).first()[0]

print("Positive class rate:", positive_rate)
print("Random PR-AUC baseline:", positive_rate)

Because reorder prediction is imbalanced, PR-AUC is more informative than accuracy. The positive class rate acts as the approximate PR-AUC of a random ranking model, so model PR-AUC should be interpreted relative to this baseline.

2. Train-validation split by user

The split is performed by `user_id`, not by row. This prevents the same user's purchase history from appearing in both training and validation data, which would make validation performance overly optimistic.

In [0]:
users = df.select("user_id").distinct()

train_users, val_users = users.randomSplit([0.8, 0.2], seed=42)

train_df = df.join(train_users, on="user_id", how="inner")
val_df = df.join(val_users, on="user_id", how="inner")

print("Train rows:", train_df.count())
print("Val rows:", val_df.count())

3. Select model features

In [0]:
feature_cols = [
    # user features
    "user_total_orders",
    "user_total_items",
    "user_unique_products",
    "user_reorder_rate",
    "user_avg_days_between_orders",
    "user_avg_cart_position",
    "user_avg_basket_size",

    # product features
    "product_total_purchases",
    "product_total_reorders",
    "product_raw_reorder_rate",
    "product_smoothed_reorder_rate",
    "product_avg_cart_position",
    "product_unique_users",
    "aisle_id",
    "department_id",

    # user-product features
    "up_times_purchased",
    "up_times_reordered",
    "up_reorder_rate",
    "up_avg_cart_position",
    "up_first_order_number",
    "up_last_order_number",
    "up_orders_since_last_purchase",
    "up_purchase_frequency",

    # order context
    "last_order_dow",
    "last_order_hour",
    "last_days_since_prior_order",
    "target_order_dow",
    "target_order_hour",
    "target_days_since_prior_order"
]

In [0]:
display(
    df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in feature_cols + ["label"]
    ])
)

## 4. Persist Train/Validation Splits

Databricks Free Edition/serverless does not support standard Spark caching in this workflow, so the train and validation splits are written to Parquet and reloaded. This makes the modeling step more stable and reproducible.

In [0]:
split_path = f"{gold_path}/model_splits"

train_df.write.mode("overwrite").parquet(f"{split_path}/train")
val_df.write.mode("overwrite").parquet(f"{split_path}/val")

train_df = spark.read.parquet(f"{split_path}/train")
val_df = spark.read.parquet(f"{split_path}/val")

print("Train rows:", train_df.count())
print("Val rows:", val_df.count())

Model 1: Logistic Regression baseline

5. Train Logistic Regression

In [0]:
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw",
    handleInvalid="keep"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=False,
    withStd=True
)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=30,
    regParam=0.01,
    elasticNetParam=0.0
)

lr_pipeline = Pipeline(stages=[assembler, scaler, lr])

In [0]:
lr_model = lr_pipeline.fit(train_df)
lr_preds = lr_model.transform(val_df)

6. Evaluate Logistic Regression

In [0]:
roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

pr_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

lr_roc_auc = roc_evaluator.evaluate(lr_preds)
lr_pr_auc = pr_evaluator.evaluate(lr_preds)

print("Logistic Regression ROC-AUC:", lr_roc_auc)
print("Logistic Regression PR-AUC:", lr_pr_auc)

The Logistic Regression baseline achieved a ROC-AUC of 0.819 and PR-AUC of 0.381 on a user-level validation split. Since reorder prediction is an imbalanced classification problem, PR-AUC is more informative than ROC-AUC. These results provide a credible baseline before training nonlinear models such as Gradient-Boosted Trees.

Model 2: GBTClassifier

7. Train GBT model

In [0]:
assembler_gbt = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=50,
    maxDepth=5,
    stepSize=0.1,
    subsamplingRate=0.8,
    seed=42
)

gbt_pipeline = Pipeline(stages=[assembler_gbt, gbt])

In [0]:
gbt_model = gbt_pipeline.fit(train_df)
gbt_preds = gbt_model.transform(val_df)

8. Evaluate GBT

In [0]:
gbt_roc_auc = roc_evaluator.evaluate(gbt_preds)
gbt_pr_auc = pr_evaluator.evaluate(gbt_preds)

print("GBT ROC-AUC:", gbt_roc_auc)
print("GBT PR-AUC:", gbt_pr_auc)

9. Extract probability score

In [0]:
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import udf

get_prob_1 = udf(lambda v: float(v[1]), DoubleType())

lr_scored = lr_preds.withColumn("score", get_prob_1(F.col("probability")))
gbt_scored = gbt_preds.withColumn("score", get_prob_1(F.col("probability")))

10. Threshold-based metrics

In [0]:
def classification_metrics(preds, threshold=0.5):
    scored = preds.withColumn(
        "pred_label",
        F.when(F.col("score") >= threshold, 1).otherwise(0)
    )

    metrics = scored.agg(
        F.sum(F.when((F.col("label") == 1) & (F.col("pred_label") == 1), 1).otherwise(0)).alias("tp"),
        F.sum(F.when((F.col("label") == 0) & (F.col("pred_label") == 1), 1).otherwise(0)).alias("fp"),
        F.sum(F.when((F.col("label") == 1) & (F.col("pred_label") == 0), 1).otherwise(0)).alias("fn"),
        F.sum(F.when((F.col("label") == 0) & (F.col("pred_label") == 0), 1).otherwise(0)).alias("tn")
    ).collect()[0]

    tp, fp, fn, tn = metrics["tp"], metrics["fp"], metrics["fn"], metrics["tn"]

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        "threshold": threshold,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [0]:
classification_metrics(lr_scored, 0.5)

In [0]:
classification_metrics(gbt_scored, 0.5)

Threshold-based metrics are useful for understanding precision-recall tradeoffs, but the final application is a recommendation problem. Therefore, Precision@K and Recall@K are more important than choosing a single global probability threshold.

11. Threshold tuning

In [0]:
thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]

lr_threshold_results = [classification_metrics(lr_scored, t) for t in thresholds]
gbt_threshold_results = [classification_metrics(gbt_scored, t) for t in thresholds]

display(spark.createDataFrame(lr_threshold_results))
display(spark.createDataFrame(gbt_threshold_results))

In [0]:
lr_threshold_df = spark.createDataFrame(lr_threshold_results).withColumn("model", F.lit("Logistic Regression"))
gbt_threshold_df = spark.createDataFrame(gbt_threshold_results).withColumn("model", F.lit("GBTClassifier"))

threshold_results_df = lr_threshold_df.unionByName(gbt_threshold_df)

threshold_results_df.write.mode("overwrite").parquet(f"{gold_path}/threshold_results")

12. Precision@K and Recall@K

In [0]:
def precision_recall_at_k(preds, k=10):
    w = Window.partitionBy("user_id").orderBy(F.desc("score"))

    top_k = (
        preds
        .withColumn("rank", F.row_number().over(w))
        .filter(F.col("rank") <= k)
    )

    user_hits = (
        top_k.groupBy("user_id")
        .agg(
            F.sum("label").alias("hits_at_k"),
            F.count("*").alias("recommended_at_k")
        )
    )

    user_actuals = (
        preds.groupBy("user_id")
        .agg(F.sum("label").alias("actual_reorders"))
    )

    metrics = (
        user_hits
        .join(user_actuals, on="user_id", how="inner")
        .withColumn("precision_at_k", F.col("hits_at_k") / F.col("recommended_at_k"))
        .withColumn(
            "recall_at_k",
            F.when(F.col("actual_reorders") > 0, F.col("hits_at_k") / F.col("actual_reorders"))
             .otherwise(None)
        )
        .agg(
            F.avg("precision_at_k").alias(f"precision_at_{k}"),
            F.avg("recall_at_k").alias(f"recall_at_{k}")
        )
    )

    return metrics.collect()[0].asDict()

In [0]:
for k in [5, 10, 15, 20]:
    print("LR", k, precision_recall_at_k(lr_scored, k))

In [0]:
for k in [5, 10, 15, 20]:
    print("GBT", k, precision_recall_at_k(gbt_scored, k))

At K=10, the GBT model achieved Precision@10 of 0.306 and Recall@10 of 0.581, meaning the model recovered about 58% of actual reordered products while recommending only 10 products per user.

13. Model comparison table

In [0]:
model_results = [
    {
        "model": "Logistic Regression",
        "roc_auc": lr_roc_auc,
        "pr_auc": lr_pr_auc,
        **precision_recall_at_k(lr_scored, 10)
    },
    {
        "model": "GBTClassifier",
        "roc_auc": gbt_roc_auc,
        "pr_auc": gbt_pr_auc,
        **precision_recall_at_k(gbt_scored, 10)
    }
]

results_df = spark.createDataFrame(model_results)
display(results_df)

GBTClassifier outperformed Logistic Regression across all evaluation metrics:

- ROC-AUC improved from 0.819 to 0.830
- PR-AUC improved from 0.381 to 0.409
- Precision@10 improved from 0.294 to 0.306
- Recall@10 improved from 0.564 to 0.581

The improvement is moderate but credible, which is expected for a strong tabular baseline with user-product interaction features.

In [0]:
results_df.write.mode("overwrite").parquet(f"{gold_path}/model_results")

14. Feature importance for GBT

In [0]:
gbt_stage = gbt_model.stages[-1]
importances = gbt_stage.featureImportances.toArray().tolist()

feature_importance_rows = list(zip(feature_cols, importances))

feature_importance_df = spark.createDataFrame(
    feature_importance_rows,
    ["feature", "importance"]
).orderBy(F.desc("importance"))

display(feature_importance_df)

Feature importance confirms that reorder prediction is primarily driven by user-product interaction history. Purchase frequency and recency were the strongest predictors, followed by product-level reorder behavior and the time since the user's previous order. This aligns with grocery shopping behavior, where routine, replenishment timing, and product loyalty are stronger signals than broad product popularity alone.

In [0]:
feature_importance_df.write.mode("overwrite").parquet(f"{gold_path}/feature_importance_gbt")

15. Save best model

In [0]:
gbt_model.write().overwrite().save(f"{model_path}/gbt_reorder_model")

GBTClassifier was selected as the final model because it improved ROC-AUC, PR-AUC, Precision@10, and Recall@10 over the Logistic Regression baseline while keeping the model pipeline scalable within Spark ML.

16. Save top recommendations

In [0]:
best_scored = gbt_scored

In [0]:
w = Window.partitionBy("user_id").orderBy(F.desc("score"))

top_recommendations = (
    best_scored
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") <= 10)
    .select(
        "user_id",
        "product_id",
        "target_order_id",
        "score",
        "rank",
        "label"
    )
)

In [0]:
products = spark.read.parquet(f"{silver_path}/product_details")

top_recommendations_named = (
    top_recommendations
    .join(
        products.select("product_id", "product_name", "aisle", "department"),
        on="product_id",
        how="left"
    )
    .select(
        "user_id",
        "target_order_id",
        "rank",
        "product_id",
        "product_name",
        "aisle",
        "department",
        "score",
        "label"
    )
    .orderBy("user_id", "rank")
)

display(top_recommendations_named.limit(50))

In [0]:
top_recommendations_named.write.mode("overwrite").parquet(
    f"{gold_path}/top10_recommendations_named"
)

In [0]:
top_recommendations.write.mode("overwrite").parquet(f"{gold_path}/top10_recommendations")

The positive class rate represents the expected PR-AUC of a random ranking model. The GBT model achieved a PR-AUC substantially above this baseline, indicating strong lift in identifying likely reorders despite class imbalance.

Summary

## Model Summary

This notebook trained reorder prediction models using a leakage-safe user-product feature table. The dataset was split by user rather than by row to prevent the same user's behavior from appearing in both training and validation sets.

Two models were evaluated:

1. Logistic Regression as an interpretable baseline
2. Gradient-Boosted Trees as the stronger nonlinear model

Because this is a recommendation problem, the final evaluation emphasizes Precision@K and Recall@K in addition to ROC-AUC and PR-AUC. The model ranks each user's previously purchased products and recommends the top K products most likely to be reordered in the next basket.

The strongest predictors are expected to come from user-product interaction history, especially purchase frequency, recency, and prior reorder behavior.